In [1]:
# Deeper CNN Architecture (VGG-inspired)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import io
from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split
import numpy as np
import os # For environment variables

# --- CNN Architecture ---
class VggCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(VggCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(9, 64, kernel_size=3, padding=1), # 9 input channels
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [2]:
# In[2]:
# --- Helper function to pre-fetch record metadata ---
def get_initial_records_and_map(db_url_str):
    """
    Connects to the database, fetches all relevant record IDs and surface groups,
    builds a label map, and splits data into training and validation record lists.
    Returns:
        train_records_list (list of (id, label_idx) tuples)
        val_records_list (list of (id, label_idx) tuples)
        label_map (dict mapping surface_group_name to label_idx)
    """
    engine = create_engine(db_url_str)
    all_records_from_db = []
    label_map = {}
    train_records_list = []
    val_records_list = []

    try:
        with engine.connect() as conn:
            result = conn.execute(text("""
                SELECT id, surface_group FROM final_balanced_trail_segments
                WHERE image_data IS NOT NULL AND surface_group IS NOT NULL
            """))
            all_records_from_db = result.fetchall()

        if not all_records_from_db:
            print("Warning: No records found in database during initial data fetch. Datasets will be empty.")
            return train_records_list, val_records_list, label_map

        # Build label_map
        unique_labels = sorted(list(set(r[1] for r in all_records_from_db)))
        label_map = {label: idx for idx, label in enumerate(unique_labels)}

        # Convert to (id, label_idx)
        all_records_with_labels = [(r[0], label_map[r[1]]) for r in all_records_from_db]

        # Split
        train_records_list, val_records_list = train_test_split(
            all_records_with_labels, test_size=0.2, random_state=42,
            stratify=[r[1] for r in all_records_with_labels] if len(all_records_with_labels) > 1 else None # Stratify if possible
        )
        print(f"Pre-fetched data: {len(train_records_list)} train, {len(val_records_list)} val records. Label map size: {len(label_map)}")

    except Exception as e:
        print(f"Error during initial data fetch: {e}")
    finally:
        if engine:
            engine.dispose() # Close all connections pooled by the engine

    return train_records_list, val_records_list, label_map


# --- Dataset Class (modified to accept pre-fetched records) ---
class TrailSegmentDataset(Dataset):
    def __init__(self, db_url, records_list, label_map, transform=None, for_stats_calc=False):
        self.db_url = db_url
        self.engine = None # Will be initialized lazily in each worker
        self.records = records_list # List of (id, label_idx)
        self.label_map = label_map
        self.transform = transform
        self.for_stats_calc = for_stats_calc # Flag for stats calculation mode

    def _get_engine(self):
        if self.engine is None:
            # print(f"Worker {os.getpid() if hasattr(os, 'getpid') else 'main'} creating new DB engine.")
            self.engine = create_engine(self.db_url, pool_size=5, max_overflow=5) # Adjust pool_size if needed for workers
        return self.engine

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        current_engine = self._get_engine()
        seg_id, label = self.records[idx]

        try:
            with current_engine.connect() as conn:
                result = conn.execute(text("SELECT image_data FROM final_balanced_trail_segments WHERE id = :id"), {"id": seg_id})
                image_bytes = result.scalar()

            if image_bytes is None:
                print(f"Warning: Image data is None for segment ID: {seg_id} (label: {label}).")
                # For stats calculation, we might want to skip. For training, this could be an error or needs a placeholder.
                if self.for_stats_calc:
                    return None, label # Signal to skip this item for stats
                else: # During actual training/validation
                    raise ValueError(f"Image data is None for segment ID: {seg_id}")

            image_array = np.load(io.BytesIO(image_bytes))
            image_array = image_array.astype(np.float32)

            if self.transform:
                image = self.transform(image_array)
            else: # Should only happen if for_stats_calc is True and transform is None
                image = image_array
            return image, label

        except Exception as e:
            print(f"Error in __getitem__ for seg_id {seg_id}: {e}")
            if self.for_stats_calc:
                return None, label # Signal to skip
            else:
                # Re-raise or return a dummy tensor to prevent crashing the whole training.
                # This depends on how you want to handle corrupted data.
                # For now, re-raising to make it visible.
                raise

In [3]:
# In[3]:
# --- Database URL ---
db_url = os.getenv('DATABASE_URL', "postgresql://markfinlay:@localhost:5432/osm")
if "localhost" in db_url and os.getenv('DATABASE_URL') is None: # Check if it's the default fallback
    print("Warning: Using default fallback DATABASE_URL pointing to localhost.")

# --- Pre-fetch initial record lists and label_map ---
# This is done ONCE in the main process
train_img_records, val_img_records, global_label_map = get_initial_records_and_map(db_url)

if not train_img_records:
    print("Halting: No training data was fetched. Check database connection and content.")
    # You might want to exit or raise an error here
    # For script continuation, provide dummy values or handle this case downstream
    channel_means = np.array([0.0] * 9, dtype=np.float32)
    channel_stds = np.array([1.0] * 9, dtype=np.float32)
else:
    # --- Calculate Mean and Std for Normalization (only on training data) ---
    print("Calculating mean and std for normalization on the training set...")
    # Create a temporary dataset instance for the training split *without normalization in transform*
    # to iterate over raw numpy arrays.
    # Pass only the training records to this temporary dataset.
    temp_train_dataset_for_stats = TrailSegmentDataset(db_url, train_img_records, global_label_map, transform=None, for_stats_calc=True)

    all_train_images_list = []
    if len(temp_train_dataset_for_stats) == 0:
        print("Error: Training dataset for stats calculation is empty (after pre-fetch).")
        channel_means = np.array([0.0] * 9, dtype=np.float32)
        channel_stds = np.array([1.0] * 9, dtype=np.float32)
    else:
        print(f"Loading {len(temp_train_dataset_for_stats)} training images to calculate normalization stats...")
        valid_images_count = 0
        for i in range(len(temp_train_dataset_for_stats)):
            image_np, _ = temp_train_dataset_for_stats[i]
            if image_np is not None: # Check if image was loaded successfully (not None from __getitem__)
                all_train_images_list.append(image_np)
                valid_images_count +=1
            if (i + 1) % 500 == 0:
                print(f"  Processed {i+1}/{len(temp_train_dataset_for_stats)} images for stats ({valid_images_count} valid)...")
        
        if not all_train_images_list:
            print("Error: No valid images were loaded for statistics calculation.")
            channel_means = np.array([0.0] * 9, dtype=np.float32)
            channel_stds = np.array([1.0] * 9, dtype=np.float32)
        else:
            all_train_images_np = np.stack(all_train_images_list, axis=0)
            print(f"Shape of stacked training images for stats: {all_train_images_np.shape}")
            channel_means = np.mean(all_train_images_np, axis=(0, 1, 2))
            channel_stds = np.std(all_train_images_np, axis=(0, 1, 2))
            epsilon = 1e-6
            channel_stds = np.maximum(channel_stds, epsilon)

    print(f"Calculated Channel Means: {channel_means.tolist()}")
    print(f"Calculated Channel Stds: {channel_stds.tolist()}")

# --- Define Final Transforms with Normalization ---
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=channel_means.tolist(), std=channel_stds.tolist()),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=channel_means.tolist(), std=channel_stds.tolist()),
])

# --- Create Final Datasets and DataLoaders ---
# Initialize datasets with pre-fetched records and the global label map
train_dataset = TrailSegmentDataset(db_url, train_img_records, global_label_map, transform=train_transform)
val_dataset = TrailSegmentDataset(db_url, val_img_records, global_label_map, transform=val_transform)


if len(train_dataset) == 0 : # Check specifically train_dataset
    print("Error: Training dataset is empty after applying transforms. Cannot proceed.")
    # Handle error: for example, by exiting or ensuring downstream code doesn't run
    model, optimizer, criterion, train_loader, val_loader = None, None, None, None, None # Prevent further execution
else:
    print(f"Training dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}") # val_dataset might be empty if train_test_split results in it

    # For MPS, num_workers > 0 can be unstable. Start with 0.
    # If using CUDA on another system, you can increase num_workers.
    num_loader_workers = 0
    use_pin_memory = False
    if torch.cuda.is_available(): # Only use pin_memory with CUDA and workers
        use_pin_memory = True 
        # num_loader_workers = 4 # Example for CUDA
    elif torch.backends.mps.is_available():
        print("MPS device detected. Setting num_workers to 0 and pin_memory to False for DataLoader for stability.")
        # num_workers > 0 with MPS is often problematic.
    
    print(f"Using num_workers = {num_loader_workers} and pin_memory = {use_pin_memory} for DataLoaders.")

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=num_loader_workers, pin_memory=use_pin_memory)
    # Ensure val_loader can handle val_dataset being empty if all data went to train
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=num_loader_workers, pin_memory=use_pin_memory) if len(val_dataset) > 0 else None

print("Global Label mapping:", global_label_map if global_label_map else "N/A")

Pre-fetched data: 98096 train, 24524 val records. Label map size: 5
Calculating mean and std for normalization on the training set...
Loading 98096 training images to calculate normalization stats...
  Processed 500/98096 images for stats (500 valid)...
  Processed 1000/98096 images for stats (1000 valid)...
  Processed 1500/98096 images for stats (1500 valid)...
  Processed 2000/98096 images for stats (2000 valid)...
  Processed 2500/98096 images for stats (2500 valid)...
  Processed 3000/98096 images for stats (3000 valid)...
  Processed 3500/98096 images for stats (3500 valid)...
  Processed 4000/98096 images for stats (4000 valid)...
  Processed 4500/98096 images for stats (4500 valid)...
  Processed 5000/98096 images for stats (5000 valid)...
  Processed 5500/98096 images for stats (5500 valid)...
  Processed 6000/98096 images for stats (6000 valid)...
  Processed 6500/98096 images for stats (6500 valid)...
  Processed 7000/98096 images for stats (7000 valid)...
  Processed 7500/9

In [4]:
# In[4]:
# --- Class Distribution and Weights ---
from collections import Counter

if len(train_dataset) > 0 and global_label_map: # Check if train_dataset is valid
    # train_labels are the indices already stored in train_img_records
    train_label_indices = [label_idx for _, label_idx in train_img_records]
    print("Training set class distribution (indices):", Counter(train_label_indices))

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Using device: {device}")

    counts = Counter(train_label_indices)
    total_samples = sum(counts.values())
    num_classes = len(global_label_map)

    weights = []
    if num_classes > 0 : # Proceed only if there are classes
        for i in range(num_classes):
            class_count = counts.get(i, 0)
            if class_count == 0:
                print(f"Warning: Class index {i} has 0 instances in the training set. Assigning weight 1.0.")
                weights.append(1.0)
            else:
                weights.append(total_samples / (num_classes * class_count))
        class_weights = torch.tensor(weights, dtype=torch.float).to(device)
    else: # No classes, no weights
        print("Warning: No classes found in label_map. Class weights cannot be calculated.")
        class_weights = None


    if class_weights is not None:
        print("Calculated class weights:")
        idx_to_label_name = {idx: name for name, idx in global_label_map.items()}
        for i, w_val in enumerate(weights): # Use w_val to avoid conflict with outer scope 'w' if any
            class_name = idx_to_label_name.get(i, f"Unknown_Class_{i}")
            print(f"Class {i} ('{class_name}'): weight = {w_val:.4f}, count = {counts.get(i, 0)}")
    else: # Handle case where class_weights could not be calculated
         print("Class weights are None.")

else:
    print("Training dataset is empty or label_map is missing, skipping class weight calculation and model training setup.")
    device = torch.device("cpu")
    model = None
    criterion = None
    optimizer = None
    class_weights = None

Training set class distribution (indices): Counter({1: 19620, 4: 19619, 0: 19619, 2: 19619, 3: 19619})
Using device: mps
Calculated class weights:
Class 0 ('asphalt'): weight = 1.0000, count = 19619
Class 1 ('grass'): weight = 1.0000, count = 19620
Class 2 ('gravel'): weight = 1.0000, count = 19619
Class 3 ('mud/dirt'): weight = 1.0000, count = 19619
Class 4 ('paved'): weight = 1.0000, count = 19619


In [5]:
# In[5]:
# --- Model, Optimizer, Criterion ---
if len(train_dataset) > 0 and global_label_map and len(global_label_map) > 0:
    model = VggCNN(num_classes=len(global_label_map)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    # Pass class_weights only if it's not None and has elements
    criterion = nn.CrossEntropyLoss(weight=class_weights if class_weights is not None and len(class_weights) > 0 else None)
else:
    print("Skipping model initialization as training dataset is empty or label_map is missing/empty.")
    model = None # Ensure model is None if not initialized
    optimizer = None
    criterion = None

In [6]:
# In[6]:
# --- Training Loop ---
# Check if model, optimizer, criterion, and train_loader are all valid
if model and optimizer and criterion and train_loader and len(train_loader) > 0:
    epoch_count = 50
    best_val_accuracy = 0.0
    patience = 10
    patience_counter = 0

    print(f"Starting training for {epoch_count} epochs on device: {device}")

    for epoch in range(epoch_count):
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Corrected loss accumulation
            running_loss += loss.item() * images.size(0)
            preds_train = outputs.argmax(dim=1)
            train_correct += (preds_train == labels).sum().item()
            train_total += labels.size(0)
            
            if (i + 1) % 100 == 0:
                print(f"  Epoch [{epoch+1}/{epoch_count}], Batch [{i+1}/{len(train_loader)}], Batch Loss: {loss.item():.4f}")

        epoch_loss = running_loss / train_total if train_total > 0 else 0
        epoch_train_accuracy = 100 * train_correct / train_total if train_total > 0 else 0
        print(f"Epoch [{epoch+1}/{epoch_count}] - Training Loss: {epoch_loss:.4f}, Training Accuracy: {epoch_train_accuracy:.2f}%")

        # Validation (only if val_loader is available and has data)
        if val_loader and len(val_loader) > 0:
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss_val = criterion(outputs, labels)
                     # Corrected loss accumulation
                    val_loss += loss_val.item() * images.size(0)
                    preds_val = outputs.argmax(dim=1)
                    val_correct += (preds_val == labels).sum().item()
                    val_total += labels.size(0)

            epoch_val_loss = val_loss / val_total if val_total > 0 else 0
            epoch_val_accuracy = 100 * val_correct / val_total if val_total > 0 else 0
            print(f"→ Epoch [{epoch+1}/{epoch_count}] - Validation Loss: {epoch_val_loss:.4f}, Validation Accuracy: {epoch_val_accuracy:.2f}%")

            if epoch_val_accuracy > best_val_accuracy:
                best_val_accuracy = epoch_val_accuracy
                patience_counter = 0
                # torch.save(model.state_dict(), 'best_vgg_model_normalized.pth')
                # print(f"Saved new best model with accuracy: {best_val_accuracy:.2f}%")
            else:
                patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs due to no improvement in validation accuracy.")
                break
        else: # No validation loader or it's empty
            print(f"→ Epoch [{epoch+1}/{epoch_count}] - Validation skipped (no validation data/loader).")
            # If no validation, you might not want early stopping or save based on training loss/accuracy.

    print("Training complete.")
else:
    print("Cannot start training: Model, optimizer, criterion, or train_loader is not properly initialized or train_loader is empty.")



Starting training for 50 epochs on device: mps
  Epoch [1/50], Batch [100/3066], Batch Loss: 1.3968
  Epoch [1/50], Batch [200/3066], Batch Loss: 1.2646
  Epoch [1/50], Batch [300/3066], Batch Loss: 1.4322
  Epoch [1/50], Batch [400/3066], Batch Loss: 1.2248
  Epoch [1/50], Batch [500/3066], Batch Loss: 1.4199
  Epoch [1/50], Batch [600/3066], Batch Loss: 1.3653
  Epoch [1/50], Batch [700/3066], Batch Loss: 1.3497
  Epoch [1/50], Batch [800/3066], Batch Loss: 1.3900
  Epoch [1/50], Batch [900/3066], Batch Loss: 1.5414
  Epoch [1/50], Batch [1000/3066], Batch Loss: 1.5562
  Epoch [1/50], Batch [1100/3066], Batch Loss: 1.3172
  Epoch [1/50], Batch [1200/3066], Batch Loss: 1.4367
  Epoch [1/50], Batch [1300/3066], Batch Loss: 1.4211
  Epoch [1/50], Batch [1400/3066], Batch Loss: 1.3399
  Epoch [1/50], Batch [1500/3066], Batch Loss: 1.4760
  Epoch [1/50], Batch [1600/3066], Batch Loss: 1.3061
  Epoch [1/50], Batch [1700/3066], Batch Loss: 1.4579
  Epoch [1/50], Batch [1800/3066], Batch Los

In [9]:
model_save_path = "VggCNN44prc.pth"
torch.save(model.state_dict(), model_save_path)

In [10]:
print(f"\n--- Loading saved model from {model_save_path} for inference/evaluation ---")

# 1. Re-create the model architecture
#    It must be the EXACT same architecture as the one you saved.
#    The number of classes must match.
loaded_model = VggCNN(num_classes=len(global_label_map))

# 2. Load the saved state dictionary
#    map_location is important if you saved the model on a GPU and want to load it on a CPU, or vice versa.
loaded_model.load_state_dict(torch.load(model_save_path, map_location=device))

# 3. Move the model to the correct device (e.g., 'mps', 'cuda', or 'cpu')
loaded_model.to(device)

# 4. Set the model to evaluation mode
#    This is crucial! It disables layers like Dropout and normalizes batch statistics correctly for inference.
loaded_model.eval()

print("Model loaded successfully and set to evaluation mode.")


--- Loading saved model from VggCNN44prc.pth for inference/evaluation ---
Model loaded successfully and set to evaluation mode.
